Import the libaries

In [1]:
!pip install gradio -q

import pandas as pd
import numpy as np
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

Load Dataset

In [2]:
df = pd.read_csv("/content/Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Check Dataset

In [3]:
print(df.shape)

(10000, 14)


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB
None


In [5]:
print(df.isnull().sum())

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


In [6]:
print(df.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

Clean Dataset

In [7]:
df = df.dropna()

# Remove ID columns if available
df = df.drop(columns=["customerID", "CustomerID", "id", "ID"], errors="ignore")

# Convert TotalCharges if present
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df = df.dropna()

Target Column

In [9]:
target_column = "Exited"
# Convert target Yes/No to 1/0
df[target_column] = df[target_column].map({
    "Yes": 1,
    "No": 0,
    "yes": 1,
    "no": 0,
    1: 1,
    0: 0
})

Encode Text Columns

In [10]:
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col].astype(str))

Split Features and Target

In [11]:
X = df.drop(target_column, axis=1)
y = df[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Scaling

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train All Algorithm for
LogisticRegresion,
RandomForestClassifier,
 GradientBoostingClassifier

In [13]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    }

    print("\n==============================")
    print(name)
    print("==============================")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Logistic Regression
Accuracy: 0.8045
Precision: 0.58
Recall: 0.14250614250614252
F1 Score: 0.22879684418145957

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.58      0.14      0.23       407

    accuracy                           0.80      2000
   macro avg       0.70      0.56      0.56      2000
weighted avg       0.77      0.80      0.75      2000

Confusion Matrix:
 [[1551   42]
 [ 349   58]]

Random Forest
Accuracy: 0.8615
Precision: 0.7927927927927928
Recall: 0.43243243243243246
F1 Score: 0.5596184419713831

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.79      0.43      0.56       407

    accuracy                           0.86      2000
   macro avg       0.83      0.70      0.74      2000
weighted avg       0.85      0.86      0.84      2000

Confusion Matrix:
 

Accuracy Comparison Table

In [14]:
result_df = pd.DataFrame(results).T
result_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.8045,0.580000,0.142506,0.228797
Random Forest,0.8615,0.792793,0.432432,0.559618
Gradient Boosting,0.8690,0.791165,0.484029,0.600610


Best Model Select

In [15]:
best_model_name = result_df["F1 Score"].idxmax()
best_model = models[best_model_name]
print("Best Model:", best_model_name)

Best Model: Gradient Boosting


Prediction Test Value

In [16]:
sample = X_test.iloc[0:1]
sample_scaled = scaler.transform(sample)
prediction = best_model.predict(sample_scaled)
print("Actual Value:", y_test.iloc[0])
print("Predicted Value:", prediction[0])

Actual Value: 0
Predicted Value: 0


UI

In [ ]:
def predict_churn(tenure, monthly_charges, total_charges):
    try:
        input_data = pd.DataFrame(
            np.zeros((1, len(X.columns))),
            columns=X.columns
        )

        if "tenure" in X.columns:
            input_data["tenure"] = tenure

        if "MonthlyCharges" in X.columns:
            input_data["MonthlyCharges"] = monthly_charges

        if "TotalCharges" in X.columns:
            input_data["TotalCharges"] = total_charges

        input_scaled = scaler.transform(input_data)

        prediction = best_model.predict(input_scaled)[0]

        if prediction == 1:
            return "⚠️ Customer is likely to Churn"
        else:
            return "✅ Customer is likely to Stay"

    except Exception as e:
        return "Error: " + str(e)


app = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Number(label="Tenure", value=12),
        gr.Number(label="Monthly Charges", value=70),
        gr.Number(label="Total Charges", value=850)
    ],
    outputs=gr.Textbox(label="Prediction Result"),
    title="📊 Customer Churn Prediction System",
    description=f"ML model using Logistic Regression, Random Forest, and Gradient Boosting. Best Model: {best_model_name}"
)

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f6682127c8ed8775a1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
